In [ ]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        raise Exception("Não é um ambiente de execução suportado.")

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q -U trl transformers peft bitsandbytes accelerate minigrid
    !pip install langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.1 MB/s eta 0:00:00


In [ ]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

Cloning into 'minigrid_benchmark'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (496/496), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 496 (delta 332), reused 445 (delta 288), pack-reused 0 (from 0)
Receiving objects: 100% (496/496), 829.58 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (332/332), done.
Repository src path: /content/minigrid_benchmark/src


In [20]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

from react_agent import ReActAgent
from wrappers import SYSTEM_PROMPT_WRAPPER_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE, OBS_TEMPLATE_ENG

import experiments_util
from experiments_util import run_and_save_experiments, wrapper1, wrapper2_with_numbers, wrapper2_without_numbers

In [21]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount("/content/drive")
    results_dir = "/content/drive/My Drive/EAD-Pesquisa-Agentes/results"

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if not os.path.exists(results_dir):
    os.makedirs(results_dir)
experiments_util.RESULTS_BASE_DIR = results_dir

Mounted at /content/drive


In [22]:
if EXECUTION_ENV == "colab":
    from google.colab import userdata
    if userdata.get("HF_TOKEN"):
        os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")

if EXECUTION_ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")

In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig

In [24]:
# 1. Configuração de Performance
MODEL_NAME = "google/gemma-4-E2B-it" # Substitua pelo modelo que você está testando
MAX_COMPLETION_LEN = 320             # Otimizado para trajetórias curtas

In [ ]:
# 2. Stopping Criteria para economizar inferência
'''class StopAtAction(StoppingCriteria):
    def __init__(self, tokenizer):
        self.stop_token = tokenizer.encode("\n", add_special_tokens=False)[-1]
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1] == self.stop_token
#'''

# 3. Carregamento do Modelo com quantização em 8-bit
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    # attn_implementation="flash_attention_2",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

In [27]:
# 4. Configuração LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

# 5. Configuração GRPO
training_args = GRPOConfig(
    output_dir="minigrid-gemma-agent",
    num_generations=4,              # Otimizado para T4 (não aumente muito!)
    max_completion_length=MAX_COMPLETION_LEN,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Simula um batch de 4
    learning_rate=2e-4,
    logging_steps=1,
    bf16=False,
    fp16=True,                      # T4 prefere fp16
    gradient_checkpointing=True,    # Economia de VRAM
)

In [41]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# 6. Instanciar modelo do hugging face
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # Atenção: necessário para contornar um bug!!!
    #stopping_criteria=StoppingCriteriaList([StopAtAction(tokenizer)]
)

lc_pipeline = HuggingFacePipeline(pipeline=hf_pipeline)
HF_CHAT_MODEL = ChatHuggingFace(llm=lc_pipeline, max_retries=2)

In [ ]:
# just a test
#r = HF_CHAT_MODEL.invoke([{"role": "human", "content": "Olá, como se diz a palavra 'português' em russo?"}])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [45]:
#r.content

'A palavra "português" em russo é **португальский** (pronuncia-se: **portugálskiy**).\n\nSe você estiver se referindo à **língua** (o idioma), você pode dizer:\n\n* **Португальский язык** (Portugaulskiy yazyk) - O idioma português.\n\nSe você estiver se referindo à **pessoa** (um falante de português), você pode dizer:\n\n* **Португалец** (Portugálets) - Um português (homem).\n* **Португалька** (Portugálka) - Uma portuguesa (mulher).<turn|>'

In [ ]:
# 7. Instanciar agente
ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),




NameError: name 'HF_CHAT_MODEL' is not defined

In [ ]:
# Função de Recompensa (Seu Wrapper entra aqui)
def reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        # Aqui você extrai a ação, roda seu env.step() e dá o reward
        # Exemplo hipotético:
        # action = parse_action(completion)
        # obs, reward, done, _ = env.step(action)
        # rewards.append(reward)
        rewards.append(0.0) # Placeholder: insira seu código de ambiente aqui
    return rewards

In [ ]:
# 8. Treinamento
# Nota: O trainer lida com a inicialização do StoppingCriteria se passado no pipeline,
# mas para controle total, prefira manter o output curto.
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=training_args,
    train_dataset=[{"prompt": "Contexto: ..."}], # Seus estados iniciais
    peft_config=peft_config,
)

trainer.train()